In [ ]:
import pandas as pd
import numpy as np
import uuid
import os

# --- 1. SETTINGS & REPRODUCIBILITY ---
N_PRODUCTS = 100
N_SESSIONS = 25000
np.random.seed(42)

# --- 2. ENHANCED PRODUCT MASTER (The Catalog) ---
def create_smart_catalog(n_products=100):
    categories = ['Audio', 'Computers', 'Peripherals', 'Cameras', 'Smart Home']
    products = []
    for i in range(n_products):
        cat = np.random.choice(categories)
        base_cost = np.random.uniform(30, 800)
        # Elasticity: How sensitive this specific SKU is to price
        elasticity = np.random.uniform(0.5, 2.5)
        inventory = np.random.randint(10, 500)

        products.append({
            'sku_id': f'ELEC-{1000+i}',
            'product_name': f'{cat} Device {i}',
            'category': cat,
            'base_cost': round(base_cost, 2),
            # Starting price with a random 1.4x to 1.9x markup
            'current_price': round(base_cost * np.random.uniform(1.4, 1.9), 2),
            'inventory_level': inventory,
            'elasticity_factor': elasticity,
            'ideal_velocity': np.random.uniform(0.02, 0.05) # SKU-specific target conversion
        })
    return pd.DataFrame(products)

# --- 3. DYNAMIC LOG GENERATION (The Clickstream) ---
def generate_gradient_logs(df_p, sessions):
    logs = []
    for _ in range(sessions):
        # Pick a random product
        prod = df_p.sample(1).iloc[0]

        # Calculate conversion probability based on price/cost and SKU 'personality'
        price_markup = prod['current_price'] / prod['base_cost']

        # Continuous conversion logic: higher price markup = lower buy probability
        # Added Gaussian noise to prevent a "perfect" mathematical fit
        buy_prob = (prod['ideal_velocity'] / (price_markup * prod['elasticity_factor']))
        buy_prob = max(0.005, buy_prob + np.random.normal(0, 0.002))

        # Funnel Logic: Cart rate is a multiple of Buy rate
        cart_prob = buy_prob * np.random.uniform(2, 5)
        view_prob = 1.0 - (buy_prob + cart_prob)

        # Ensure probabilities don't exceed 1.0 or fall below 0.0
        p_dist = np.clip([view_prob, cart_prob, buy_prob], 0, 1)
        p_dist /= p_dist.sum() # Re-normalize

        logs.append({
            'session_id': str(uuid.uuid4())[:8], # Required by processing.py
            'sku_id': prod['sku_id'],
            'duration_sec': np.random.randint(10, 400),
            'action': np.random.choice([0, 1, 2], p=p_dist) # 0=View, 1=Cart, 2=Buy
        })
    return pd.DataFrame(logs)

# --- 4. EXECUTION & FILE SAVING ---
# Generate the data
df_prod_master = create_smart_catalog(N_PRODUCTS)
df_clickstream = generate_gradient_logs(df_prod_master, N_SESSIONS)

# Save to CSV (Required for SageMaker Processing Input)
# These filenames must match what your processing.py expects
df_prod_master.to_csv('product_master.csv', index=False)
df_clickstream.to_csv('clickstream_normal.csv', index=False)

print(f"✅ SUCCESS: Created {len(df_prod_master)} SKUs and {len(df_clickstream)} sessions.")
print("📁 Files saved: 'product_master.csv' and 'clickstream_normal.csv'")
print("🚀 You can now upload these to your S3 bucket for the MLOps pipeline.")